## Document Ingestion, OCR & Structuring

### What this notebook does
1. Reads every supported file from `data/raw/{pdfs,office,images,html,json,metadata}`
2. Extracts text with **PyMuPDF** (native PDF text) and falls back to **EasyOCR** only on pages/images that need it
3. Extracts structured content (headings, paragraphs, tables, lists) from **DOCX** with `python-docx`
4. Cleans text (whitespace, unicode, Arabic normalization, repeated headers/footers, dedup) without over-cleaning
5. Builds rich **metadata** (sha256, language, page counts, timestamps, etc.)
6. Writes **one page-structured JSON per document** — never a single flattened string
7. Produces `processing_report.json`, `failed_documents.json`, `processing_statistics.json`
8. Logs everything, and **never crashes the whole run** because of one bad file

### Architecture

```
DocumentProcessor (base/dispatcher)
 ├── PDFProcessor        → PyMuPDF + selective OCR per page
 ├── OfficeProcessor      → python-docx (DOCX/TXT)
 ├── OCRProcessor         → EasyOCR (en + ar) for images / scanned pages
 ├── ImagePreprocessor    → resize / grayscale / contrast / threshold / deskew / denoise
 ├── TextCleaner          → normalization & de-duplication
 ├── MetadataExtractor    → per-document metadata dict
 └── PipelineManager      → orchestrates everything, handles errors, writes reports
```

### Output contract
Every processed document becomes a single JSON file shaped like:

```json
{
  "document_id": "...",
  "title": "...",
  "source_file": "...",
  "document_type": "pdf",
  "language": "en",
  "pages": [
    {"page": 1, "blocks": [{"type": "heading", "text": "..."}, {"type": "paragraph", "text": "..."}]}
  ],
  "metadata": { "...": "..." }
}
```



## 1. Environment Setup

Installs all dependencies needed for parsing, OCR, and image preprocessing.
Run once per Colab session. Safe to re-run (idempotent).


In [ ]:

# ============================================================
# 1.1 Install dependencies (Colab-safe, quiet)
# ============================================================
!pip install -q pymupdf python-docx easyocr opencv-python-headless pillow \
    tqdm arabic-reshaper python-bidi unicodedata2 langdetect
print("✅ Dependencies installed.")


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 981.5/981.5 kB 25.0 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 25.7/25.7 MB 84.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 253.0/253.0 kB 24.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.9/2.9 MB 119.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 296.2/296.2 kB 31.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 531.9/531.9 kB 45.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 180.7/180.7 kB 19.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 978.2/978.2 kB 76.0 MB/s eta 0:00:00
✅ Dependencies installed.


In [ ]:

# ============================================================
# 1.2 Mount Google Drive
# ============================================================
from google.colab import drive
drive.mount('/content/drive')


Mounted at /content/drive


## 2. Imports

In [ ]:

from __future__ import annotations

import os
import re
import io
import json
import time
import hashlib
import logging
import unicodedata
import traceback
from dataclasses import dataclass, field, asdict
from datetime import datetime, timezone
from pathlib import Path
from typing import Any, Optional, Union

import cv2
import numpy as np
from PIL import Image
import fitz  # PyMuPDF
import docx  # python-docx
from tqdm.auto import tqdm

try:
    from langdetect import detect as _langdetect_detect
    _HAS_LANGDETECT = True
except ImportError:
    _HAS_LANGDETECT = False

print("✅ Imports ready.")


✅ Imports ready.



## 3. Configuration

All paths are centralized here. Change `PROJECT_ROOT` if your Drive layout differs.
The notebook **only reads** from `RAW_*` paths and **only writes** under `PROCESSED_ROOT`.


In [ ]:

# ============================================================
# 3.1 Path configuration
# ============================================================

PROJECT_ROOT = Path("/content/drive/MyDrive/SmartUniversityAssistant")

RAW_ROOT       = PROJECT_ROOT / "data" / "raw"
RAW_PDFS       = RAW_ROOT / "pdfs"
RAW_OFFICE     = RAW_ROOT / "office"
RAW_IMAGES     = RAW_ROOT / "images"
RAW_HTML       = RAW_ROOT / "html"
RAW_JSON       = RAW_ROOT / "json"
RAW_METADATA   = RAW_ROOT / "metadata"

PROCESSED_ROOT     = PROJECT_ROOT / "data" / "processed"
PROCESSED_DOCS      = PROCESSED_ROOT / "documents"
PROCESSED_METADATA  = PROCESSED_ROOT / "metadata"
PROCESSED_LOGS      = PROCESSED_ROOT / "logs"

for p in [PROCESSED_DOCS, PROCESSED_METADATA, PROCESSED_LOGS]:
    p.mkdir(parents=True, exist_ok=True)

# ============================================================
# 3.2 Pipeline configuration
# ============================================================

@dataclass
class PipelineConfig:
    """Central knobs for the whole pipeline. Tune here, not inside classes."""

    # OCR
    ocr_languages: tuple = ("en", "ar")
    ocr_gpu: bool = True
    min_native_text_chars_per_page: int = 40   # OCR that page
    min_ocr_text_chars: int = 15                # discard OCR

    # Image preprocessing
    target_max_dimension: int = 2000
    contrast_clip_limit: float = 2.0
    apply_deskew: bool = True
    apply_denoise: bool = True

    # Text cleaning
    max_repeated_line_ratio: float = 0.6  # a line appearing on >60% of pages = header/footer

    # Supported extensions per family
    pdf_exts: tuple = (".pdf",)
    office_exts: tuple = (".docx", ".txt")
    image_exts: tuple = (".png", ".jpg", ".jpeg")
    json_exts: tuple = (".json",)
    html_exts: tuple = (".html", ".htm")

    # HTML snapshots are supported by HTMLProcessor but OFF by default (per project decision).
    # Flip this to True (and add RAW_HTML to RAW_DIRS_TO_PROCESS) to enable them later.
    enable_html_processing: bool = False

    # Misc
    encoding: str = "utf-8"

CONFIG = PipelineConfig()
print(CONFIG)


PipelineConfig(ocr_languages=('en', 'ar'), ocr_gpu=True, min_native_text_chars_per_page=40, min_ocr_text_chars=15, target_max_dimension=2000, contrast_clip_limit=2.0, apply_deskew=True, apply_denoise=True, max_repeated_line_ratio=0.6, pdf_exts=('.pdf',), office_exts=('.docx', '.txt'), image_exts=('.png', '.jpg', '.jpeg'), json_exts=('.json',), html_exts=('.html', '.htm'), enable_html_processing=False, encoding='utf-8')



## 4. Structured Logging

Every file processed, every OCR fallback, every failure is logged both to console
and to a persistent log file under `data/processed/logs/`, so the run is fully auditable.


In [ ]:

# ============================================================
# 4.1 Logger setup
# ============================================================

def build_logger(log_dir: Path) -> logging.Logger:
    """Create a logger that writes to both console and a timestamped file."""
    log_dir.mkdir(parents=True, exist_ok=True)
    run_id = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")
    log_path = log_dir / f"pipeline_run_{run_id}.log"

    logger = logging.getLogger("pipeline")
    logger.setLevel(logging.INFO)
    logger.handlers.clear()

    fmt = logging.Formatter("%(asctime)s | %(levelname)-8s | %(message)s")

    fh = logging.FileHandler(log_path, encoding="utf-8")
    fh.setFormatter(fmt)
    logger.addHandler(fh)

    ch = logging.StreamHandler()
    ch.setFormatter(fmt)
    logger.addHandler(ch)

    logger.info(f"Logging initialized. Log file: {log_path}")
    return logger

LOGGER = build_logger(PROCESSED_LOGS)


2026-07-27 21:01:11,229 | INFO     | Logging initialized. Log file: /content/drive/MyDrive/SmartUniversityAssistant/data/processed/logs/pipeline_run_20260727T210111Z.log
INFO:member1_pipeline:Logging initialized. Log file: /content/drive/MyDrive/SmartUniversityAssistant/data/processed/logs/pipeline_run_20260727T210111Z.log



## 5. `TextCleaner`

Light-touch cleaning that **preserves structure**. It never deletes headings or
paragraph boundaries — it only removes noise (extra whitespace, duplicated
boilerplate, repeated running headers/footers, common OCR artifacts).


In [ ]:

class TextCleaner:
    """Normalizes and de-noises extracted text without destroying structure.

    Designed to be reusable across PDF, DOCX, and OCR text sources.
    """

    _ARABIC_NORMALIZE_MAP = {
        "\u0623": "\u0627",  # alef with hamza above -> plain alef
        "\u0625": "\u0627",  # alef with hamza below -> plain alef
        "\u0622": "\u0627",  # alef with madda above -> plain alef
        "\u0629": "\u0647",  # taa marbuta -> haa
        "\u0649": "\u064a",  # alef maqsura -> yaa
    }

    _OCR_FIX_MAP = {
        "|": "I",
        "0CR": "OCR",
        "``": '"',
        "''": '"',
    }

    def __init__(self, config: PipelineConfig = CONFIG):
        self.config = config

    def normalize_unicode(self, text: str) -> str:
        """NFC-normalize unicode so equivalent glyphs compare equal."""
        return unicodedata.normalize("NFC", text)

    def normalize_arabic(self, text: str) -> str:
        """Normalize common Arabic character variants (alef/hamza forms, taa marbuta, alef maqsura)."""
        for src, dst in self._ARABIC_NORMALIZE_MAP.items():
            text = text.replace(src, dst)
        # Remove tatweel/kashida elongation character
        text = text.replace("\u0640", "")
        return text

    def basic_ocr_corrections(self, text: str) -> str:
        """Fix a handful of very common OCR misrecognitions."""
        for wrong, right in self._OCR_FIX_MAP.items():
            text = text.replace(wrong, right)
        return text

    def remove_duplicate_whitespace(self, text: str) -> str:
        text = re.sub(r"[ \t]+", " ", text)
        text = re.sub(r"\n{3,}", "\n\n", text)
        return text

    def remove_empty_lines(self, text: str) -> str:
        lines = [ln for ln in text.split("\n") if ln.strip() != ""]
        return "\n".join(lines)

    def remove_duplicated_paragraphs(self, paragraphs: list[str]) -> list[str]:
        """Drop exact-duplicate paragraphs while preserving first-seen order."""
        seen = set()
        result = []
        for p in paragraphs:
            key = p.strip()
            if key and key not in seen:
                seen.add(key)
                result.append(p)
            elif not key:
                result.append(p)
        return result

    def detect_repeated_headers_footers(self, pages_text: list[str]) -> set[str]:
        """Find lines that repeat across most pages (running headers/footers) so they can be stripped."""
        if len(pages_text) < 3:
            return set()
        line_counts: dict[str, int] = {}
        for page_text in pages_text:
            first_last = set()
            lines = [l.strip() for l in page_text.split("\n") if l.strip()]
            if lines:
                first_last.add(lines[0])
                first_last.add(lines[-1])
            for ln in first_last:
                line_counts[ln] = line_counts.get(ln, 0) + 1

        threshold = max(2, int(len(pages_text) * self.config.max_repeated_line_ratio))
        return {ln for ln, cnt in line_counts.items() if cnt >= threshold and len(ln) < 120}

    def strip_lines(self, text: str, lines_to_remove: set[str]) -> str:
        if not lines_to_remove:
            return text
        kept = [ln for ln in text.split("\n") if ln.strip() not in lines_to_remove]
        return "\n".join(kept)

    def clean(self, text: str, is_arabic_hint: bool = False) -> str:
        """Full single-string cleaning pipeline (used for paragraph/OCR fragments)."""
        if not text:
            return ""
        text = self.normalize_unicode(text)
        if is_arabic_hint:
            text = self.normalize_arabic(text)
        text = self.basic_ocr_corrections(text)
        text = self.remove_duplicate_whitespace(text)
        text = self.remove_empty_lines(text)
        return text.strip()


TEXT_CLEANER = TextCleaner(CONFIG)
print("✅ TextCleaner ready.")


✅ TextCleaner ready.



## 6. `ImagePreprocessor`

Prepares an image for OCR: resize → grayscale → contrast enhancement (CLAHE) →
adaptive threshold → deskew → denoise. Used both for standalone images and for
rasterized PDF pages that need OCR.


In [ ]:

class ImagePreprocessor:
    """Applies a configurable OCR-preprocessing pipeline to an image (numpy array, BGR or gray)."""

    def __init__(self, config: PipelineConfig = CONFIG):
        self.config = config

    def resize(self, img: np.ndarray) -> np.ndarray:
        h, w = img.shape[:2]
        max_dim = max(h, w)
        if max_dim <= self.config.target_max_dimension:
            return img
        scale = self.config.target_max_dimension / max_dim
        return cv2.resize(img, (int(w * scale), int(h * scale)), interpolation=cv2.INTER_AREA)

    def to_grayscale(self, img: np.ndarray) -> np.ndarray:
        if len(img.shape) == 2:
            return img
        return cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)

    def enhance_contrast(self, gray: np.ndarray) -> np.ndarray:
        clahe = cv2.createCLAHE(clipLimit=self.config.contrast_clip_limit, tileGridSize=(8, 8))
        return clahe.apply(gray)

    def threshold(self, gray: np.ndarray) -> np.ndarray:
        return cv2.adaptiveThreshold(
            gray, 255, cv2.ADAPTIVE_THRESH_GAUSSIAN_C, cv2.THRESH_BINARY, 31, 11
        )

    def deskew(self, img: np.ndarray) -> np.ndarray:
        """Estimate and correct small rotational skew using the minAreaRect of text pixels."""
        coords = np.column_stack(np.where(img < 255))
        if coords.shape[0] < 20:
            return img
        angle = cv2.minAreaRect(coords)[-1]
        if angle < -45:
            angle = -(90 + angle)
        else:
            angle = -angle
        if abs(angle) < 0.3:  # not worth rotating
            return img
        (h, w) = img.shape[:2]
        center = (w // 2, h // 2)
        M = cv2.getRotationMatrix2D(center, angle, 1.0)
        return cv2.warpAffine(img, M, (w, h), flags=cv2.INTER_CUBIC, borderMode=cv2.BORDER_REPLICATE)

    def denoise(self, img: np.ndarray) -> np.ndarray:
        return cv2.fastNlMeansDenoising(img, h=10)

    def preprocess(self, img: np.ndarray) -> np.ndarray:
        """Full pipeline: resize -> grayscale -> contrast -> threshold -> deskew -> denoise."""
        img = self.resize(img)
        gray = self.to_grayscale(img)
        gray = self.enhance_contrast(gray)
        thresh = self.threshold(gray)
        if self.config.apply_deskew:
            thresh = self.deskew(thresh)
        if self.config.apply_denoise:
            thresh = self.denoise(thresh)
        return thresh


IMAGE_PREPROCESSOR = ImagePreprocessor(CONFIG)
print("✅ ImagePreprocessor ready.")


✅ ImagePreprocessor ready.



## 7. `OCRProcessor`

Thin, reusable wrapper around **EasyOCR** (English + Arabic). Applies the
`ImagePreprocessor` pipeline first, runs OCR, and discards results that are too
short to be useful (so we never emit empty/near-empty documents).


In [ ]:

import easyocr

class OCRProcessor:
    """Runs EasyOCR over a preprocessed image and returns cleaned text, or None if not useful."""

    _reader = None  # lazy singleton — EasyOCR model load is expensive

    def __init__(self, config: PipelineConfig = CONFIG, preprocessor: Optional[ImagePreprocessor] = None):
        self.config = config
        self.preprocessor = preprocessor or IMAGE_PREPROCESSOR

    @classmethod
    def get_reader(cls, config: PipelineConfig = CONFIG):
        if cls._reader is None:
            LOGGER.info(f"Loading EasyOCR reader for languages={config.ocr_languages} (first call only)...")
            cls._reader = easyocr.Reader(list(config.ocr_languages), gpu=config.ocr_gpu)
        return cls._reader

    def run_on_array(self, img_bgr_or_gray: np.ndarray) -> Optional[str]:
        """Preprocess + OCR a numpy image array. Returns None if result is too short to be useful."""
        processed = self.preprocessor.preprocess(img_bgr_or_gray)
        reader = self.get_reader(self.config)
        try:
            results = reader.readtext(processed, detail=0, paragraph=True)
        except Exception as e:
            LOGGER.warning(f"EasyOCR failed on an image: {e}")
            return None

        text = "\n".join(r.strip() for r in results if r and r.strip())
        if len(text.strip()) < self.config.min_ocr_text_chars:
            return None
        return text

    def run_on_path(self, image_path: Union[str, Path]) -> Optional[str]:
        img = cv2.imread(str(image_path))
        if img is None:
            LOGGER.warning(f"Could not read image for OCR: {image_path}")
            return None
        return self.run_on_array(img)

    def run_on_pil(self, pil_image: Image.Image) -> Optional[str]:
        arr = cv2.cvtColor(np.array(pil_image.convert("RGB")), cv2.COLOR_RGB2BGR)
        return self.run_on_array(arr)


OCR_PROCESSOR = OCRProcessor(CONFIG, IMAGE_PREPROCESSOR)
print("✅ OCRProcessor ready (model loads lazily on first use).")


✅ OCRProcessor ready (model loads lazily on first use).



## 8. `MetadataExtractor`

Builds the metadata block attached to every processed document: id, hashes,
language, page counts, crawl provenance (read from Stage 1's own metadata files
when available), and processing status.


In [ ]:

class MetadataExtractor:
    """Builds the `metadata` object for a processed document."""

    def __init__(self, raw_metadata_dir: Path = RAW_METADATA):
        self.raw_metadata_dir = raw_metadata_dir
        self._crawl_metadata_cache: Optional[dict] = None

    def _load_crawl_metadata(self) -> dict:
        """Stage 1 may have dropped a metadata JSON (or several) describing source_url/category
        per file. We index it once by filename for fast lookup; missing file -> empty index."""
        if self._crawl_metadata_cache is not None:
            return self._crawl_metadata_cache

        index: dict[str, dict] = {}
        if self.raw_metadata_dir.exists():
            for meta_file in self.raw_metadata_dir.glob("*.json"):
                try:
                    data = json.loads(meta_file.read_text(encoding="utf-8"))
                    entries = data if isinstance(data, list) else [data]
                    for entry in entries:
                        fname = entry.get("filename") or entry.get("source_file") or entry.get("file")
                        if fname:
                            index[Path(fname).name] = entry
                except Exception as e:
                    LOGGER.warning(f"Could not parse crawl metadata file {meta_file}: {e}")
        self._crawl_metadata_cache = index
        return index

    @staticmethod
    def compute_sha256(file_path: Path) -> str:
        h = hashlib.sha256()
        with open(file_path, "rb") as f:
            for chunk in iter(lambda: f.read(8192), b""):
                h.update(chunk)
        return h.hexdigest()

    @staticmethod
    def make_document_id(file_path: Path, sha256: str) -> str:
        return f"{file_path.stem}_{sha256[:10]}"

    @staticmethod
    def detect_language(sample_text: str) -> str:
        if not sample_text or not sample_text.strip():
            return "unknown"
        if _HAS_LANGDETECT:
            try:
                return _langdetect_detect(sample_text[:2000])
            except Exception:
                pass
        # Cheap fallback: proportion of Arabic-range characters
        arabic_chars = sum(1 for c in sample_text if "\u0600" <= c <= "\u06FF")
        return "ar" if arabic_chars > len(sample_text) * 0.3 else "en"

    def build(
        self,
        file_path: Path,
        document_type: str,
        title: str,
        page_count: int,
        language: str,
        ocr_used: bool,
        processing_status: str,
        sha256: Optional[str] = None,
        source_url: Optional[str] = None,
        crawl_category: Optional[str] = None,
        domain: Optional[str] = None,
        extra: Optional[dict] = None,
    ) -> dict:
        """Build the metadata block.

        `source_url` / `crawl_category` / `domain` can be supplied directly by a processor
        that already found them inside the document itself (e.g. JSONProcessor reading the
        crawler's own JSON schema, or HTMLProcessor reading a snapshot). When not supplied,
        we fall back to Stage 1's separate `data/raw/metadata/` index looked up by filename.
        `sha256` can be passed in to reuse a hash already computed upstream (e.g. by
        PipelineManager for duplicate detection) instead of re-hashing the file.
        `extra` lets a processor attach source-specific metadata fields on top of the base
        contract below (e.g. JSONProcessor forwards the crawler's own doc_id/content_hash/
        fetched_at/status_code/content_type/word_count/outlinks/pdf_links/office_links/
        image_links). It is additive only -- when omitted, the returned dict is identical
        to before, so PDF/Office/Image documents are completely unaffected.
        """
        crawl_index = self._load_crawl_metadata()
        crawl_entry = crawl_index.get(file_path.name, {})
        sha256 = sha256 or self.compute_sha256(file_path)

        metadata = {
            "document_id": self.make_document_id(file_path, sha256),
            "title": title,
            "source_file": str(file_path),
            "source_url": source_url or crawl_entry.get("source_url"),
            "document_type": document_type,
            "language": language,
            "page_count": page_count,
            "crawl_category": crawl_category or crawl_entry.get("category") or crawl_entry.get("crawl_category"),
            "domain": domain or crawl_entry.get("domain"),
            "sha256": sha256,
            "processing_date": datetime.now(timezone.utc).isoformat(),
            "ocr_used": ocr_used,
            "processing_status": processing_status,
        }
        if extra:
            metadata.update(extra)
        return metadata


METADATA_EXTRACTOR = MetadataExtractor(RAW_METADATA)
print("✅ MetadataExtractor ready.")


✅ MetadataExtractor ready.



## 9. `PDFProcessor`

Uses **PyMuPDF** as the primary parser. For each page:
- if native text length ≥ `min_native_text_chars_per_page` → use extracted text directly
- otherwise → rasterize **only that page** and send it through OCR

This avoids OCR-ing entire documents unnecessarily while still catching scanned pages
inside otherwise-digital PDFs.


In [ ]:

class PDFProcessor:
    """Extracts page-structured content from a PDF, using native text with selective OCR fallback."""

    def __init__(
        self,
        config: PipelineConfig = CONFIG,
        ocr: OCRProcessor = OCR_PROCESSOR,
        cleaner: TextCleaner = TEXT_CLEANER,
    ):
        self.config = config
        self.ocr = ocr
        self.cleaner = cleaner

    def _page_to_image_array(self, page: "fitz.Page", zoom: float = 2.0) -> np.ndarray:
        mat = fitz.Matrix(zoom, zoom)
        pix = page.get_pixmap(matrix=mat)
        img = np.frombuffer(pix.samples, dtype=np.uint8).reshape(pix.height, pix.width, pix.n)
        if pix.n == 4:
            img = cv2.cvtColor(img, cv2.COLOR_RGBA2BGR)
        elif pix.n == 3:
            img = cv2.cvtColor(img, cv2.COLOR_RGB2BGR)
        return img

    def _split_into_blocks(self, raw_text: str, is_arabic_hint: bool) -> list[dict]:
        """Turn raw page text into heading/paragraph blocks.

        Heuristic: short, all-caps-ish or title-like standalone lines followed by
        blank lines are treated as headings; everything else becomes paragraphs.
        """
        blocks = []
        raw_paragraphs = [p for p in re.split(r"\n\s*\n", raw_text) if p.strip()]
        raw_paragraphs = self.cleaner.remove_duplicated_paragraphs(raw_paragraphs)

        for para in raw_paragraphs:
            cleaned = self.cleaner.clean(para, is_arabic_hint=is_arabic_hint)
            if not cleaned:
                continue
            is_heading = (
                len(cleaned) < 90
                and "\n" not in cleaned
                and (cleaned.isupper() or cleaned.istitle() or cleaned.endswith(":"))
            )
            blocks.append({"type": "heading" if is_heading else "paragraph", "text": cleaned})
        return blocks

    def process(self, file_path: Path) -> dict:
        """Returns (pages, ocr_used, page_count, sample_text_for_lang_detection)."""
        doc = fitz.open(str(file_path))
        pages_out = []
        raw_page_texts = []
        ocr_used_any = False

        for page_index in range(len(doc)):
            page = doc[page_index]
            native_text = page.get_text("text") or ""

            if len(native_text.strip()) >= self.config.min_native_text_chars_per_page:
                page_text = native_text
                used_ocr_here = False
            else:
                img_arr = self._page_to_image_array(page)
                ocr_text = self.ocr.run_on_array(img_arr)
                if ocr_text is None:
                    LOGGER.info(
                        f"{file_path.name} page {page_index + 1}: too little text even after OCR — skipping page content."
                    )
                    page_text = ""
                    used_ocr_here = True
                else:
                    page_text = ocr_text
                    used_ocr_here = True

            if used_ocr_here:
                ocr_used_any = True

            raw_page_texts.append(page_text)

        # detect repeated headers/footers across the whole doc, then strip + block-ify per page
        boilerplate_lines = self.cleaner.detect_repeated_headers_footers(raw_page_texts)
        sample_for_lang = "\n".join(raw_page_texts[:3])
        is_arabic_hint = MetadataExtractor.detect_language(sample_for_lang) == "ar"

        for i, page_text in enumerate(raw_page_texts):
            page_text = self.cleaner.strip_lines(page_text, boilerplate_lines)
            blocks = self._split_into_blocks(page_text, is_arabic_hint)
            pages_out.append({"page": i + 1, "blocks": blocks})

        doc.close()
        return {
            "pages": pages_out,
            "ocr_used": ocr_used_any,
            "page_count": len(raw_page_texts),
            "sample_text": sample_for_lang,
        }


PDF_PROCESSOR = PDFProcessor(CONFIG, OCR_PROCESSOR, TEXT_CLEANER)
print("✅ PDFProcessor ready.")


✅ PDFProcessor ready.



## 10. `OfficeProcessor`

Handles **DOCX** (via `python-docx`, preserving headings/paragraphs/tables/lists)
and plain **TXT** files. DOCX and TXT don't have a native concept of "pages", so
each document is emitted as a single logical page (`page: 1`) containing the full
block structure — this keeps the output schema uniform for next step.



In [ ]:

class OfficeProcessor:
    """Extracts structured content from DOCX and TXT files."""

    def __init__(self, cleaner: TextCleaner = TEXT_CLEANER):
        self.cleaner = cleaner

    def _docx_block_type(self, paragraph: "docx.text.paragraph.Paragraph") -> str:
        style_name = (paragraph.style.name or "").lower()
        if "heading" in style_name or "title" in style_name:
            return "heading"
        if "list" in style_name or paragraph.text.strip().startswith(("-", "•", "*")):
            return "list_item"
        return "paragraph"

    def _table_to_block(self, table: "docx.table.Table") -> dict:
        rows = []
        for row in table.rows:
            rows.append([cell.text.strip() for cell in row.cells])
        return {"type": "table", "text": json.dumps(rows, ensure_ascii=False)}

    def process_docx(self, file_path: Path) -> dict:
        document = docx.Document(str(file_path))
        blocks = []
        is_arabic_hint = False
        sample_chunks = []

        for para in document.paragraphs:
            text = para.text.strip()
            if not text:
                continue
            cleaned = self.cleaner.clean(text)
            if not cleaned:
                continue
            sample_chunks.append(cleaned)
            blocks.append({"type": self._docx_block_type(para), "text": cleaned})

        for table in document.tables:
            blocks.append(self._table_to_block(table))

        sample_text = "\n".join(sample_chunks[:20])
        if MetadataExtractor.detect_language(sample_text) == "ar":
            is_arabic_hint = True
            blocks = [
                {**b, "text": self.cleaner.normalize_arabic(b["text"])} if b["type"] != "table" else b
                for b in blocks
            ]

        return {
            "pages": [{"page": 1, "blocks": blocks}],
            "ocr_used": False,
            "page_count": 1,
            "sample_text": sample_text,
        }

    def process_txt(self, file_path: Path) -> dict:
        raw = file_path.read_text(encoding="utf-8", errors="ignore")
        paragraphs = [p for p in re.split(r"\n\s*\n", raw) if p.strip()]
        paragraphs = self.cleaner.remove_duplicated_paragraphs(paragraphs)

        blocks = []
        for p in paragraphs:
            cleaned = self.cleaner.clean(p)
            if cleaned:
                blocks.append({"type": "paragraph", "text": cleaned})

        sample_text = "\n".join(b["text"] for b in blocks[:20])
        return {
            "pages": [{"page": 1, "blocks": blocks}],
            "ocr_used": False,
            "page_count": 1,
            "sample_text": sample_text,
        }

    def process(self, file_path: Path) -> dict:
        if file_path.suffix.lower() == ".docx":
            return self.process_docx(file_path)
        elif file_path.suffix.lower() == ".txt":
            return self.process_txt(file_path)
        raise ValueError(f"Unsupported office file type: {file_path.suffix}")


OFFICE_PROCESSOR = OfficeProcessor(TEXT_CLEANER)
print("✅ OfficeProcessor ready.")


✅ OfficeProcessor ready.



## 11. Standalone Image Handler

For files that are already single images (`.png`, `.jpg`, `.jpeg`) rather than
PDF pages — same OCR path, wrapped to emit the same `pages/blocks` schema.


In [ ]:

class StandaloneImageProcessor:
    """Runs OCR on a standalone image file and emits the standard pages/blocks schema."""

    def __init__(self, ocr: OCRProcessor = OCR_PROCESSOR, cleaner: TextCleaner = TEXT_CLEANER):
        self.ocr = ocr
        self.cleaner = cleaner

    def process(self, file_path: Path) -> dict:
        text = self.ocr.run_on_path(file_path)
        if text is None:
            # Not enough usable text — emit an empty-but-valid document rather than throwing.
            return {
                "pages": [{"page": 1, "blocks": []}],
                "ocr_used": True,
                "page_count": 1,
                "sample_text": "",
                "skipped_low_text": True,
            }

        is_arabic_hint = MetadataExtractor.detect_language(text) == "ar"
        cleaned = self.cleaner.clean(text, is_arabic_hint=is_arabic_hint)
        paragraphs = [p for p in cleaned.split("\n") if p.strip()]

        blocks = [{"type": "paragraph", "text": p} for p in paragraphs]
        return {
            "pages": [{"page": 1, "blocks": blocks}],
            "ocr_used": True,
            "page_count": 1,
            "sample_text": cleaned,
            "skipped_low_text": False,
        }


IMAGE_DOC_PROCESSOR = StandaloneImageProcessor(OCR_PROCESSOR, TEXT_CLEANER)
print("✅ StandaloneImageProcessor ready.")


✅ StandaloneImageProcessor ready.



## 12. `JSONProcessor`

Reads the ~845 crawler-generated JSON files under `data/raw/json/` — by far the largest
knowledge source in this dataset. The crawler emits one **fixed** schema for every file
(`doc_id`, `url`, `domain`, `category`, `title`, `headings`, `paragraphs`, `tables`,
`full_text`, `word_count`, `outlinks`, `pdf_links`, `office_links`, `image_links`,
`content_hash`, `fetched_at`, `status_code`, `content_type`), so this processor reads those
fields directly instead of guessing key names. It builds `heading` / `paragraph` / `table`
blocks from the structured fields (falling back to `full_text` only if those are missing),
reuses the existing `TextCleaner`, and emits the exact same `pages`/`blocks` contract as
every other processor. No OCR, no HTML parsing — pure JSON → structured text.


In [ ]:

class JSONProcessor:
    """Extracts structured content from Stage-1 crawler-generated JSON files (data/raw/json/).

    The crawler's schema is FIXED across the entire dataset (see project README / sample
    files), so this processor reads the known fields directly rather than probing multiple
    candidate key names:

        doc_id, url, domain, category, title, headings, paragraphs, tables, full_text,
        word_count, outlinks, pdf_links, office_links, image_links, content_hash,
        fetched_at, status_code, content_type

    Structure is preserved by building separate heading / paragraph / table blocks instead
    of collapsing everything into `full_text`. `full_text` is used only as a fallback when
    `headings`, `paragraphs`, and `tables` are all missing or empty for a given file.

    Reuses TextCleaner for all text normalization and returns the same pages/blocks dict
    shape as PDFProcessor and OfficeProcessor, so DocumentProcessor can treat it identically.
    """

    def __init__(self, cleaner: TextCleaner = TEXT_CLEANER):
        self.cleaner = cleaner

    @staticmethod
    def _heading_text(item: Any) -> Optional[str]:
        """A heading entry is normally a plain string. Some crawls emit richer
        {"level": 2, "text": "..."} objects for the same field -- handle both without
        introducing any new *field names* to search for."""
        if isinstance(item, str):
            return item
        if isinstance(item, dict):
            return item.get("text")
        return None

    @staticmethod
    def _table_to_rows(table: Any) -> Optional[list[list[str]]]:
        """A table entry is a list of rows, each row a list of cell strings. Some crawls
        represent a row as {"col_a": "...", "col_b": "..."} instead of a plain list --
        handle both, same reasoning as `_heading_text` above."""
        if not isinstance(table, list) or not table:
            return None
        rows: list[list[str]] = []
        for row in table:
            if isinstance(row, list):
                rows.append([str(cell).strip() for cell in row])
            elif isinstance(row, dict):
                rows.append([str(v).strip() for v in row.values()])
        return rows or None

    def _structured_blocks(self, raw: dict, is_arabic_hint: bool) -> list[dict]:
        """Build blocks straight from the crawler's own headings/paragraphs/tables fields."""
        blocks: list[dict] = []

        for heading in raw.get("headings") or []:
            text = self._heading_text(heading)
            if not text:
                continue
            cleaned = self.cleaner.clean(str(text), is_arabic_hint=is_arabic_hint)
            if cleaned:
                blocks.append({"type": "heading", "text": cleaned})

        paragraphs = [p for p in (raw.get("paragraphs") or []) if isinstance(p, str) and p.strip()]
        paragraphs = self.cleaner.remove_duplicated_paragraphs(paragraphs)
        for para in paragraphs:
            cleaned = self.cleaner.clean(para, is_arabic_hint=is_arabic_hint)
            if cleaned:
                blocks.append({"type": "paragraph", "text": cleaned})

        for table in raw.get("tables") or []:
            rows = self._table_to_rows(table)
            if rows:
                blocks.append({"type": "table", "text": json.dumps(rows, ensure_ascii=False)})

        return blocks

    def _fallback_blocks_from_full_text(self, full_text: str, is_arabic_hint: bool) -> list[dict]:
        """Only used when headings/paragraphs/tables are all missing or empty."""
        blocks: list[dict] = []
        paragraphs = [p for p in re.split(r"\n\s*\n", full_text or "") if p.strip()]
        paragraphs = self.cleaner.remove_duplicated_paragraphs(paragraphs)
        for para in paragraphs:
            cleaned = self.cleaner.clean(para, is_arabic_hint=is_arabic_hint)
            if cleaned:
                blocks.append({"type": "paragraph", "text": cleaned})
        return blocks

    def process(self, file_path: Path) -> dict:
        """Parse one crawler JSON file into the standard pages/blocks dict.

        Raises ValueError if the file isn't even a JSON object, so PipelineManager can log
        it as a failure without crashing the whole run.
        """
        raw = json.loads(file_path.read_text(encoding="utf-8", errors="ignore"))
        if not isinstance(raw, dict):
            raise ValueError(f"Unrecognized JSON schema (expected an object): {file_path.name}")

        title_val = raw.get("title")
        full_text = raw.get("full_text") or ""
        probe_text = (str(title_val) if title_val else "") + "\n" + full_text
        is_arabic_hint = MetadataExtractor.detect_language(probe_text[:1000]) == "ar"

        blocks = self._structured_blocks(raw, is_arabic_hint)
        used_fallback = False
        if not blocks and full_text.strip():
            blocks = self._fallback_blocks_from_full_text(full_text, is_arabic_hint)
            used_fallback = True
            LOGGER.info(
                f"{file_path.name}: headings/paragraphs/tables all empty, fell back to full_text."
            )

        title_clean = None
        if title_val:
            title_clean = self.cleaner.clean(str(title_val), is_arabic_hint=is_arabic_hint)
            if title_clean:
                # The full_text fallback commonly repeats the title as its own first
                # paragraph (crawlers often prepend it). Drop that duplicate before we
                # insert the title as its own heading block, so it isn't shown twice.
                if blocks and blocks[0]["type"] == "paragraph" and blocks[0]["text"].strip() == title_clean.strip():
                    blocks.pop(0)
                blocks.insert(0, {"type": "heading", "text": title_clean})

        if not blocks:
            LOGGER.warning(f"{file_path.name}: no usable content in headings/paragraphs/tables/full_text.")

        sample_text = "\n".join(b["text"] for b in blocks[:20])

        # Prefer the crawler's own word_count; only recompute if it's missing, since the
        # crawler's figure is presumably measured against its own full_text extraction.
        word_count = raw.get("word_count")
        if word_count is None:
            all_text = "\n".join(b["text"] for b in blocks if b["type"] != "table")
            word_count = len(all_text.split()) if all_text else 0

        return {
            "pages": [{"page": 1, "blocks": blocks}],
            "ocr_used": False,
            "page_count": 1,
            "sample_text": sample_text,
            "title_override": title_clean,
            "source_url": raw.get("url"),
            "crawl_category": raw.get("category"),
            "domain": raw.get("domain"),
            "used_full_text_fallback": used_fallback,
            # Forwarded as-is into MetadataExtractor.build(extra=...) -- see DocumentProcessor.
            "extra_metadata": {
                "crawler_doc_id": raw.get("doc_id"),
                "content_hash": raw.get("content_hash"),
                "fetched_at": raw.get("fetched_at"),
                "status_code": raw.get("status_code"),
                "content_type": raw.get("content_type"),
                "word_count": word_count,
                "outlinks": raw.get("outlinks") or [],
                "pdf_links": raw.get("pdf_links") or [],
                "office_links": raw.get("office_links") or [],
                "image_links": raw.get("image_links") or [],
            },
        }


JSON_PROCESSOR = JSONProcessor(TEXT_CLEANER)
print("✅ JSONProcessor ready.")


✅ JSONProcessor ready.



## 13. `HTMLProcessor` (optional — disabled by default)

**New, but not wired into the default run.** Extracts visible text + `<title>` from raw
HTML snapshots under `data/raw/html/`, using only the Python standard library
(`html.parser`) — no BeautifulSoup, no extra dependency. `DocumentProcessor` is capable of
routing `.html`/`.htm` files to it, gated behind `PipelineConfig.enable_html_processing`
(default `False`). To turn it on later: set `CONFIG.enable_html_processing = True` and add
`RAW_HTML` to `RAW_DIRS_TO_PROCESS` in the run cell — nothing else needs to change.


In [ ]:

from html.parser import HTMLParser


class _VisibleTextExtractor(HTMLParser):
    """Minimal, dependency-free HTML text extractor: strips script/style/nav/etc. boilerplate
    and pulls out <title> and the remaining visible text. Intentionally simple — this is a
    fallback path, not the primary ingestion route (JSONProcessor already covers the
    equivalent content for the 845 crawled pages)."""

    _SKIP_TAGS = {"script", "style", "noscript", "header", "footer", "nav", "aside"}

    def __init__(self):
        super().__init__()
        self.title = ""
        self._in_title = False
        self._skip_depth = 0
        self.chunks: list[str] = []

    def handle_starttag(self, tag, attrs):
        if tag == "title":
            self._in_title = True
        if tag in self._SKIP_TAGS:
            self._skip_depth += 1

    def handle_endtag(self, tag):
        if tag == "title":
            self._in_title = False
        if tag in self._SKIP_TAGS and self._skip_depth > 0:
            self._skip_depth -= 1

    def handle_data(self, data):
        if self._in_title:
            self.title += data
        elif self._skip_depth == 0:
            stripped = data.strip()
            if stripped:
                self.chunks.append(stripped)


class HTMLProcessor:
    """OPTIONAL extension for raw HTML snapshots. Not enabled by default — see
    PipelineConfig.enable_html_processing and DocumentProcessor._document_type.
    """

    def __init__(self, cleaner: TextCleaner = TEXT_CLEANER):
        self.cleaner = cleaner

    def process(self, file_path: Path) -> dict:
        raw_html = file_path.read_text(encoding="utf-8", errors="ignore")
        extractor = _VisibleTextExtractor()
        extractor.feed(raw_html)
        extractor.close()

        title = extractor.title.strip()
        probe = (title + " " + " ".join(extractor.chunks[:5]))[:1000]
        is_arabic_hint = MetadataExtractor.detect_language(probe) == "ar"

        paragraphs = self.cleaner.remove_duplicated_paragraphs(extractor.chunks)
        blocks = []

        title_clean = None
        if title:
            title_clean = self.cleaner.clean(title, is_arabic_hint=is_arabic_hint)
            if title_clean:
                blocks.append({"type": "heading", "text": title_clean})

        for para in paragraphs:
            cleaned = self.cleaner.clean(para, is_arabic_hint=is_arabic_hint)
            if cleaned:
                blocks.append({"type": "paragraph", "text": cleaned})

        sample_text = "\n".join(b["text"] for b in blocks[:20])
        return {
            "pages": [{"page": 1, "blocks": blocks}],
            "ocr_used": False,
            "page_count": 1,
            "sample_text": sample_text,
            "title_override": title_clean,
        }


HTML_PROCESSOR = HTMLProcessor(TEXT_CLEANER)
print("✅ HTMLProcessor ready (constructed, but not routed to by default — see CONFIG.enable_html_processing).")


✅ HTMLProcessor ready (constructed, but not routed to by default — see CONFIG.enable_html_processing).



## 14. `DocumentProcessor`

Single entry point that looks at a file's extension and routes it to the right
specialized processor, then wraps the result together with metadata into the
final document JSON shape.


In [ ]:

class DocumentProcessor:
    """Routes a single file to the correct processor and assembles the final document dict."""

    def __init__(
        self,
        config: PipelineConfig = CONFIG,
        pdf_processor: PDFProcessor = PDF_PROCESSOR,
        office_processor: OfficeProcessor = OFFICE_PROCESSOR,
        image_processor: StandaloneImageProcessor = IMAGE_DOC_PROCESSOR,
        json_processor: JSONProcessor = JSON_PROCESSOR,
        html_processor: HTMLProcessor = HTML_PROCESSOR,
        metadata_extractor: MetadataExtractor = METADATA_EXTRACTOR,
    ):
        self.config = config
        self.pdf_processor = pdf_processor
        self.office_processor = office_processor
        self.image_processor = image_processor
        self.json_processor = json_processor
        self.html_processor = html_processor
        self.metadata_extractor = metadata_extractor

    def _document_type(self, file_path: Path) -> str:
        ext = file_path.suffix.lower()
        if ext in self.config.pdf_exts:
            return "pdf"
        if ext == ".docx":
            return "docx"
        if ext == ".txt":
            return "txt"
        if ext in self.config.image_exts:
            return "image"
        if ext in self.config.json_exts:
            return "json"
        if ext in self.config.html_exts:
            return "html"
        return "unsupported"

    def is_supported(self, file_path: Path) -> bool:
        doc_type = self._document_type(file_path)
        if doc_type == "unsupported":
            return False
        if doc_type == "html" and not self.config.enable_html_processing:
            # HTMLProcessor exists and DocumentProcessor can route to it, but it stays
            # off by default per project decision — treat as unsupported until flipped on.
            return False
        return True

    def process(self, file_path: Path) -> dict:
        """Process a single file end-to-end. Raises on failure (caught by PipelineManager)."""
        doc_type = self._document_type(file_path)

        if doc_type == "pdf":
            result = self.pdf_processor.process(file_path)
        elif doc_type in ("docx", "txt"):
            result = self.office_processor.process(file_path)
        elif doc_type == "image":
            result = self.image_processor.process(file_path)
        elif doc_type == "json":
            result = self.json_processor.process(file_path)
        elif doc_type == "html":
            if not self.config.enable_html_processing:
                raise ValueError("HTML processing is disabled (CONFIG.enable_html_processing = False)")
            result = self.html_processor.process(file_path)
        else:
            raise ValueError(f"Unsupported file type: {file_path.suffix}")

        language = MetadataExtractor.detect_language(result.get("sample_text", ""))

        # JSON/HTML sources often carry their own title (page <title>, crawler "title" field);
        # prefer that over a filename-derived guess whenever the processor found one.
        title_override = result.get("title_override")
        title = title_override if title_override else file_path.stem.replace("_", " ").replace("-", " ").strip().title()

        metadata = self.metadata_extractor.build(
            file_path=file_path,
            document_type=doc_type,
            title=title,
            page_count=result["page_count"],
            language=language,
            ocr_used=result["ocr_used"],
            processing_status="success",
            source_url=result.get("source_url"),
            crawl_category=result.get("crawl_category"),
            domain=result.get("domain"),
            extra=result.get("extra_metadata"),
        )

        return {
            "document_id": metadata["document_id"],
            "title": title,
            "source_file": str(file_path),
            "document_type": doc_type,
            "language": language,
            "pages": result["pages"],
            "metadata": metadata,
        }


DOCUMENT_PROCESSOR = DocumentProcessor(
    CONFIG, PDF_PROCESSOR, OFFICE_PROCESSOR, IMAGE_DOC_PROCESSOR, JSON_PROCESSOR, HTML_PROCESSOR, METADATA_EXTRACTOR
)
print("✅ DocumentProcessor ready (pdf / docx / txt / image / json supported; html implemented but disabled).")


✅ DocumentProcessor ready (pdf / docx / txt / image / json supported; html implemented but disabled).



## 15. `PipelineManager`

Walks all raw input folders, calls `DocumentProcessor` on every supported file,
**never lets one bad file stop the run**, writes one JSON per document, and
produces the three summary reports.


In [ ]:

class PipelineManager:
    """Orchestrates discovery, processing, error handling, output writing, and reporting."""

    def __init__(
        self,
        processor: DocumentProcessor = DOCUMENT_PROCESSOR,
        output_docs_dir: Path = PROCESSED_DOCS,
        output_metadata_dir: Path = PROCESSED_METADATA,
        output_logs_dir: Path = PROCESSED_LOGS,
        logger: logging.Logger = LOGGER,
    ):
        self.processor = processor
        self.output_docs_dir = output_docs_dir
        self.output_metadata_dir = output_metadata_dir
        self.output_logs_dir = output_logs_dir
        self.logger = logger

        self.successes: list[dict] = []
        self.failures: list[dict] = []
        self.skipped: list[dict] = []
        self.duplicates: list[dict] = []
        self._seen_hashes: dict[str, str] = {}  # sha256 -> path of the first file we saw with it

    def discover_files(self, root_dirs: list[Path]) -> list[Path]:
        """Recursively find every candidate file under the given raw-data directories."""
        all_files = []
        for root in root_dirs:
            if not root.exists():
                self.logger.warning(f"Raw directory does not exist, skipping: {root}")
                continue
            all_files.extend(sorted(p for p in root.rglob("*") if p.is_file()))
        return all_files

    def _write_document(self, document: dict) -> Path:
        out_path = self.output_docs_dir / f"{document['document_id']}.json"
        out_path.write_text(json.dumps(document, ensure_ascii=False, indent=2), encoding="utf-8")

        # also mirror the metadata block alone, for quick indexing / a vector DB
        meta_path = self.output_metadata_dir / f"{document['document_id']}.json"
        meta_path.write_text(json.dumps(document["metadata"], ensure_ascii=False, indent=2), encoding="utf-8")
        return out_path

    def run(self, root_dirs: list[Path]) -> None:
        files = self.discover_files(root_dirs)
        self.logger.info(f"Discovered {len(files)} candidate files across {len(root_dirs)} raw folders.")

        for file_path in tqdm(files, desc="Processing documents", unit="file"):
            start = time.time()

            if not self.processor.is_supported(file_path):
                self.logger.info(f"Skipping unsupported file: {file_path}")
                self.skipped.append({"file": str(file_path), "reason": "unsupported_extension"})
                continue

            # Duplicate detection: hash first (cheap) so we never waste OCR/parse time on a
            # file we've already processed byte-for-byte under a different name/location.
            try:
                file_hash = MetadataExtractor.compute_sha256(file_path)
            except Exception as e:
                self.logger.warning(f"Could not hash {file_path} for duplicate check: {e}")
                file_hash = None

            if file_hash is not None and file_hash in self._seen_hashes:
                original_path = self._seen_hashes[file_hash]
                self.logger.warning(
                    f"Duplicate detected (sha256={file_hash[:12]}...): {file_path} "
                    f"is identical to already-processed {original_path}. Skipping."
                )
                self.duplicates.append(
                    {"file": str(file_path), "duplicate_of": original_path, "sha256": file_hash}
                )
                continue
            if file_hash is not None:
                self._seen_hashes[file_hash] = str(file_path)

            try:
                document = self.processor.process(file_path)
                # Reuse the hash we already computed above instead of re-hashing inside
                # MetadataExtractor.build.
                if file_hash is not None:
                    document["metadata"]["sha256"] = file_hash
                self._write_document(document)
                elapsed = time.time() - start
                self.logger.info(
                    f"Processed OK: {file_path.name} "
                    f"(pages={document['metadata']['page_count']}, "
                    f"ocr_used={document['metadata']['ocr_used']}, "
                    f"{elapsed:.2f}s)"
                )
                self.successes.append(
                    {
                        "file": str(file_path),
                        "document_id": document["document_id"],
                        "pages": document["metadata"]["page_count"],
                        "ocr_used": document["metadata"]["ocr_used"],
                        "processing_time_seconds": round(elapsed, 3),
                    }
                )
            except Exception as e:
                # CRITICAL: never let one bad file kill the whole run.
                elapsed = time.time() - start
                self.logger.error(f"FAILED: {file_path} -> {e}")
                self.logger.debug(traceback.format_exc())
                self.failures.append(
                    {
                        "file": str(file_path),
                        "error": str(e),
                        "traceback": traceback.format_exc(),
                        "processing_time_seconds": round(elapsed, 3),
                    }
                )

        self._write_reports()

    def _write_reports(self) -> None:
        report = {
            "run_timestamp": datetime.now(timezone.utc).isoformat(),
            "total_files_seen": len(self.successes) + len(self.failures) + len(self.skipped) + len(self.duplicates),
            "successful": len(self.successes),
            "failed": len(self.failures),
            "skipped_unsupported": len(self.skipped),
            "duplicates_skipped": len(self.duplicates),
            "successes": self.successes,
            "skipped": self.skipped,
            "duplicates": self.duplicates,
        }
        (self.output_logs_dir / "processing_report.json").write_text(
            json.dumps(report, ensure_ascii=False, indent=2), encoding="utf-8"
        )

        (self.output_logs_dir / "failed_documents.json").write_text(
            json.dumps(self.failures, ensure_ascii=False, indent=2), encoding="utf-8"
        )

        total_pages = sum(s["pages"] for s in self.successes)
        ocr_docs = sum(1 for s in self.successes if s["ocr_used"])
        avg_time = (
            round(sum(s["processing_time_seconds"] for s in self.successes) / len(self.successes), 3)
            if self.successes
            else 0
        )
        stats = {
            "total_documents_processed": len(self.successes),
            "total_documents_failed": len(self.failures),
            "total_documents_skipped": len(self.skipped),
            "total_duplicates_skipped": len(self.duplicates),
            "total_pages_extracted": total_pages,
            "documents_requiring_ocr": ocr_docs,
            "average_processing_time_seconds": avg_time,
            "success_rate_percent": (
                round(100 * len(self.successes) / max(1, len(self.successes) + len(self.failures)), 2)
            ),
        }
        (self.output_logs_dir / "processing_statistics.json").write_text(
            json.dumps(stats, ensure_ascii=False, indent=2), encoding="utf-8"
        )

        self.logger.info(f"Report written: {report['successful']} ok / {report['failed']} failed / "
                          f"{report['skipped_unsupported']} skipped / {report['duplicates_skipped']} duplicates.")
        self.logger.info(f"Statistics: {stats}")


PIPELINE = PipelineManager(
    processor=DOCUMENT_PROCESSOR,
    output_docs_dir=PROCESSED_DOCS,
    output_metadata_dir=PROCESSED_METADATA,
    output_logs_dir=PROCESSED_LOGS,
    logger=LOGGER,
)
print("✅ PipelineManager ready.")


✅ PipelineManager ready.



## 16. Run the Pipeline

This is the only cell that actually touches your Drive data. It processes every
supported file under `pdfs/`, `office/`, and `images/`. (`html/` and `json/` are
left for a dedicated processor if your Stage 1 crawler produces them — see the
note below.)


In [ ]:

RAW_DIRS_TO_PROCESS = [RAW_JSON, RAW_PDFS, RAW_OFFICE, RAW_IMAGES]

PIPELINE.run(RAW_DIRS_TO_PROCESS)


2026-07-27 21:01:38,892 | INFO     | Discovered 1270 candidate files across 4 raw folders.
INFO:member1_pipeline:Discovered 1270 candidate files across 4 raw folders.


Processing documents:   0%|          | 0/1270 [00:00<?, ?file/s]

2026-07-27 21:01:40,736 | INFO     | Processed OK: 0029ae65943eda2f.json (pages=1, ocr_used=False, 1.83s)
INFO:member1_pipeline:Processed OK: 0029ae65943eda2f.json (pages=1, ocr_used=False, 1.83s)
2026-07-27 21:01:41,073 | INFO     | Processed OK: 0040472cc177908d.json (pages=1, ocr_used=False, 0.33s)
INFO:member1_pipeline:Processed OK: 0040472cc177908d.json (pages=1, ocr_used=False, 0.33s)
2026-07-27 21:01:41,346 | INFO     | Processed OK: 0092e825b91cc23b.json (pages=1, ocr_used=False, 0.27s)
INFO:member1_pipeline:Processed OK: 0092e825b91cc23b.json (pages=1, ocr_used=False, 0.27s)
2026-07-27 21:01:41,687 | INFO     | Processed OK: 00a804e29234cc32.json (pages=1, ocr_used=False, 0.34s)
INFO:member1_pipeline:Processed OK: 00a804e29234cc32.json (pages=1, ocr_used=False, 0.34s)
2026-07-27 21:01:41,982 | INFO     | Processed OK: 00cf5a5ea75ca24a.json (pages=1, ocr_used=False, 0.29s)
INFO:member1_pipeline:Processed OK: 00cf5a5ea75ca24a.json (pages=1, ocr_used=False, 0.29s)
2026-07-27 21:0

Progress: |██████████████████████████████████████████████████| 100.0% Complete

Progress: |██████████████████████████████████████████████████| 100.0% Complete

2026-07-27 21:03:48,662 | INFO     | Processed OK: e337a22676d54277__app_uploads_2026_02_Final_SOLE_Handbook_Last_Update_2_6_26_p.pdf (pages=59, ocr_used=True, 16.86s)
INFO:member1_pipeline:Processed OK: e337a22676d54277__app_uploads_2026_02_Final_SOLE_Handbook_Last_Update_2_6_26_p.pdf (pages=59, ocr_used=True, 16.86s)
2026-07-27 21:03:49,017 | INFO     | Processed OK: e35288b313979308__mit_procedures_academic_performance_grades_academic_performa.pdf (pages=1, ocr_used=False, 0.35s)
INFO:member1_pipeline:Processed OK: e35288b313979308__mit_procedures_academic_performance_grades_academic_performa.pdf (pages=1, ocr_used=False, 0.35s)
2026-07-27 21:03:49,329 | INFO     | Processed OK: e449b6b8d6d61782__sites_default_files_2018_05_tuition_07_08_pdf.pdf (pages=10, ocr_used=False, 0.31s)
INFO:member1_pipeline:Processed OK: e449b6b8d6d61782__sites_default_files_2018_05_tuition_07_08_pdf.pdf (pages=10, ocr_used=False, 0.31s)
2026-07-27 21:03:49,702 | INFO     | Processed OK: e4949547977d0491__


> **Note on `html/`:** `data/raw/json/` (the largest source, ~845 files) is now fully
> processed via `JSONProcessor`. `data/raw/html/` raw snapshots are intentionally **not**
> included in `RAW_DIRS_TO_PROCESS` above — `HTMLProcessor` is implemented and
> `DocumentProcessor` already knows how to route `.html`/`.htm` files to it, but it stays off
> until you explicitly opt in. To enable it: set `CONFIG.enable_html_processing = True` and
> add `RAW_HTML` to `RAW_DIRS_TO_PROCESS` — no other code needs to change.


## 17. Review Statistics & Sample Output

In [ ]:

stats_path = PROCESSED_LOGS / "processing_statistics.json"
print(json.dumps(json.loads(stats_path.read_text(encoding="utf-8")), ensure_ascii=False, indent=2))


{
  "total_documents_processed": 1264,
  "total_documents_failed": 0,
  "total_documents_skipped": 1,
  "total_duplicates_skipped": 5,
  "total_pages_extracted": 2143,
  "documents_requiring_ocr": 128,
  "average_processing_time_seconds": 0.193,
  "success_rate_percent": 100.0
}


In [ ]:

# Show one processed document as a sanity check
sample_files = sorted(PROCESSED_DOCS.glob("*.json"))
if sample_files:
    sample = json.loads(sample_files[0].read_text(encoding="utf-8"))
    print(f"Sample document: {sample_files[0].name}\n")
    print(json.dumps(sample, ensure_ascii=False, indent=2)[:3000])
else:
    print("No documents were processed yet — check data/raw/ paths and processing_report.json")


Sample document: 0029ae65943eda2f_e148e9885c.json

{
  "document_id": "0029ae65943eda2f_e148e9885c",
  "title": "Academic Calendar I MIT Registrar",
  "source_file": "/content/drive/MyDrive/SmartUniversityAssistant/data/raw/json/0029ae65943eda2f.json",
  "document_type": "json",
  "language": "en",
  "pages": [
    {
      "page": 1,
      "blocks": [
        {
          "type": "heading",
          "text": "Academic Calendar I MIT Registrar"
        },
        {
          "type": "heading",
          "text": "Academic Calendar"
        },
        {
          "type": "heading",
          "text": "September 2026"
        },
        {
          "type": "paragraph",
          "text": "Search"
        },
        {
          "type": "paragraph",
          "text": "Main Menu"
        },
        {
          "type": "paragraph",
          "text": "Category(-)OrientationThesisAcademic deadlinesDegree list deadlinesTuition & financial aid deadlinesExamsHolidaysMeetings"
        },
        {
    


## 18. Handoff

`data/processed/documents/*.json` — one page-structured document per source file, ready for:
- Recursive / sliding-window **chunking** (iterate `pages[*].blocks[*].text`, using `type` to keep headings attached to their section)
- **Embedding generation** (chunk-level, with `document_id` + `page` carried through for **citations**)

`data/processed/metadata/*.json` — same `document_id` keys, for fast metadata lookups without opening the full document.

`data/processed/logs/` — `processing_report.json`, `failed_documents.json`, `processing_statistics.json` for QA / grading evidence.


In [ ]:
# ============================================================
# 19. JSONProcessor Update -- Run Report
# ============================================================
# Reads what was actually written to disk by this run and summarizes:
#   - overall pipeline totals (from processing_statistics.json)
#   - how many docs came from the crawler JSON files specifically
#   - how much of each doc's content is structured (heading/paragraph/table)
#     vs. how many fell back to full_text
#   - how consistently the crawler metadata fields made it through

stats_path = PROCESSED_LOGS / "processing_statistics.json"
overall_stats = json.loads(stats_path.read_text(encoding="utf-8")) if stats_path.exists() else {}

json_docs = []
for doc_path in sorted(PROCESSED_DOCS.glob("*.json")):
    doc = json.loads(doc_path.read_text(encoding="utf-8"))
    if doc.get("document_type") == "json":
        json_docs.append(doc)

n_json = len(json_docs)
block_counts = {"heading": 0, "paragraph": 0, "table": 0}
fallback_count = 0
empty_count = 0
word_counts = []
metadata_field_presence = {
    "crawler_doc_id": 0, "content_hash": 0, "fetched_at": 0, "status_code": 0,
    "content_type": 0, "word_count": 0, "outlinks": 0, "pdf_links": 0,
    "office_links": 0, "image_links": 0,
}

for doc in json_docs:
    meta = doc.get("metadata", {})
    blocks = doc["pages"][0]["blocks"] if doc.get("pages") else []

    if not blocks:
        empty_count += 1
    for b in blocks:
        block_counts[b["type"]] = block_counts.get(b["type"], 0) + 1

    if meta.get("used_full_text_fallback"):
        fallback_count += 1

    wc = meta.get("word_count")
    if isinstance(wc, (int, float)):
        word_counts.append(wc)

    for field in metadata_field_presence:
        value = meta.get(field)
        # outlinks/pdf_links/etc. are lists -- "present" means non-empty for those,
        # for the rest it just means not None.
        if isinstance(value, list):
            if value:
                metadata_field_presence[field] += 1
        elif value is not None:
            metadata_field_presence[field] += 1

print("=" * 60)
print("JSONProcessor Update -- Run Report")
print("=" * 60)

print("\nOverall pipeline (all file types):")
for k, v in overall_stats.items():
    print(f"  {k}: {v}")

print(f"\nJSON-sourced documents (data/raw/json/): {n_json}")
print(f"  Documents that used the full_text fallback : {fallback_count} "
      f"({(100 * fallback_count / n_json):.1f}%)" if n_json else "  (none processed)")
print(f"  Documents with no usable content at all    : {empty_count}")

print(f"\nBlocks written across all JSON documents:")
for block_type, count in block_counts.items():
    print(f"  {block_type:10s}: {count}")

if word_counts:
    print(f"\nword_count (from crawler / recomputed on fallback):")
    print(f"  min={min(word_counts)}  max={max(word_counts)}  "
          f"avg={sum(word_counts) / len(word_counts):.1f}")

print(f"\nCrawler metadata field coverage ({n_json} docs):")
for field, count in metadata_field_presence.items():
    pct = (100 * count / n_json) if n_json else 0
    print(f"  {field:20s}: {count}/{n_json} ({pct:.1f}%)")

print("=" * 60)

JSONProcessor Update -- Run Report

Overall pipeline (all file types):
  total_documents_processed: 1264
  total_documents_failed: 0
  total_documents_skipped: 1
  total_duplicates_skipped: 5
  total_pages_extracted: 2143
  documents_requiring_ocr: 128
  average_processing_time_seconds: 0.193
  success_rate_percent: 100.0

JSON-sourced documents (data/raw/json/): 1000
  Documents that used the full_text fallback : 0 (0.0%)
  Documents with no usable content at all    : 0

Blocks written across all JSON documents:
  heading   : 12488
  paragraph : 41430
  table     : 4943

word_count (from crawler / recomputed on fallback):
  min=2  max=41696  avg=684.2

Crawler metadata field coverage (1000 docs):
  crawler_doc_id      : 1000/1000 (100.0%)
  content_hash        : 1000/1000 (100.0%)
  fetched_at          : 1000/1000 (100.0%)
  status_code         : 1000/1000 (100.0%)
  content_type        : 1000/1000 (100.0%)
  word_count          : 1000/1000 (100.0%)
  outlinks            : 945/1000 (9